In [18]:
import numpy as np
import pandas as pd

In [19]:
df = pd.read_csv("Final_SHAP_Analysis.csv")

In [20]:
ACTION_MAP = {
    "monthly_cost": {
        "action": "ارائه بسته تخفیف ویژه (فقط برای HIGH RISK با الگوی مصرف بالا) - توصیه نمی‌شود مگر در کنار انگیزه ارزشی",
        "campaign": "Conditional_Discount_Only_With_Value"
    },

    "gen_2G": {
        "action": "پیشنهاد ارتقا به 4G/5G از طریق خط سریع اختصاصی (MEDIUM RISK)",
        "campaign": "Network_Upgrade_via_Express_Lane",
    },

    "gen_3G": {
        "action": "پیشنهاد ارتقا به 4G/5G از طریق خط سریع اختصاصی (MEDIUM RISK)",
        "campaign": "Network_Upgrade_via_Express_Lane",
    },

    "operator_app": {
        "action": "اعطای پاداش نصب اپلیکیشن به صورت امتیاز عضویت در باشگاه قهرمانان (LOW RISK)",
        "campaign": "App_Engagement_Heroes_Points",
    },

    "volte": {
        "action": "فعال‌سازی رایگان VoLTE به عنوان بخشی از بسته عذرخواهی برای CRITICAL RISK",
        "campaign": "VoLTE_As_Apology_Bundle",
    },

    "sim_history": {
        "action": "اجرای کمپین خوش‌آمدگویی پیشگیرانه ۲۴ ساعته برای مشتریان جدید (CRITICAL RISK)",
        "campaign": "Welcome_Campaign_24h_Guardian",
    },

    "superapp": {
        "action": "ارائه سرویس دیجیتال رایگان (مثلاً ۱۰ گیگ دیتای خودکار ماهانه) برای HIGH RISK دارای مصرف بالا",
        "campaign": "Digital_Bundle_Automatic_Topup",
    }
}

In [ ]:
def generate_recommendation(row):

    reasons = str(row["Top_Churn_Reasons"]).lower()

    recommendations = []
    campaigns = []


    for key, value in ACTION_MAP.items():

        if key in reasons:
            recommendations.append(value["action"])
            campaigns.append(value["campaign"])

    # if no reason

    if len(recommendations) == 0:
        recommendations.append("برنامه وفادارسازی عمومی")
        campaigns.append("General Loyalty")

    return pd.Series({
        "Recommended_Action":  " | ".join(list(set(recommendations))),
        "Campaign_Type":  " | ".join(list(set(campaigns)))})

In [25]:
recommendation_results = df.apply(generate_recommendation, axis=1)

df = pd.concat([df, recommendation_results], axis=1)

In [26]:
def customer_priority(row):
    prob = row["Churn_Probability"]

    if prob >= 0.9:
        return "Critical"
    elif prob >= 0.75:
        return "Very High"
    elif prob >= 0.5:
        return "High"
    else:
        return "Normal"

df["Customer_Priority"] = df.apply(customer_priority, axis=1)

In [ ]:
df.to_csv(

    "Final_Recommendation_System.csv",
    index=False,
    encoding="utf-8-sig"
)